In [1]:
# ==========================================================
# CELL 1 — IMPORTS AND PATHS
# ==========================================================

import pandas as pd
import sqlite3
from pathlib import Path

pd.set_option("display.max_columns", 100)

# Check where Jupyter is currently running
print("Current working directory:")
print(Path.cwd())

Current working directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\notebooks


In [2]:
# ==========================================================
# CELL 2 — FIND PROJECT DIRECTORIES
# ==========================================================

CURRENT_DIR = Path.cwd()

print("Current directory:", CURRENT_DIR)

# If notebook is inside notebooks/, project is one level above
if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SQL_DIR = BASE_DIR / "sql"

print("Base directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)
print("SQL directory:", SQL_DIR)

print("\nProcessed files:")

if PROCESSED_DIR.exists():
    for file in PROCESSED_DIR.glob("*.csv"):
        print(" -", file.name)
else:
    print("WARNING: processed folder not found")

Current directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\notebooks
Base directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics
Processed directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed
SQL directory: c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql

Processed files:
 - accuracy_by_match_result.csv
 - group_standings.csv
 - matches.csv
 - matches_features.csv
 - match_prediction_results.csv
 - model_comparison.csv
 - players_features.csv
 - qualified_teams.csv
 - stadiums_cleaned.csv
 - teams_features.csv


In [3]:
# ==========================================================
# CELL 3 — CREATE SQL FOLDER
# ==========================================================

SQL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("SQL folder ready:")
print(SQL_DIR)

SQL folder ready:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql


In [4]:
# ==========================================================
# CELL 4 — LOAD FEATURE-ENGINEERED DATA
# ==========================================================

players = pd.read_csv(
    PROCESSED_DIR / "players_features.csv"
)

teams = pd.read_csv(
    PROCESSED_DIR / "teams_features.csv"
)

matches = pd.read_csv(
    PROCESSED_DIR / "matches_features.csv"
)

print("Players:", players.shape)
print("Teams:", teams.shape)
print("Matches:", matches.shape)

Players: (1248, 80)
Teams: (48, 137)
Matches: (104, 53)


In [5]:
# ==========================================================
# CELL 3 — CREATE SQLITE DATABASE
# ==========================================================

DATABASE_PATH = SQL_DIR / "football_analytics.db"

connection = sqlite3.connect(
    DATABASE_PATH
)

print("SQLite database created:")
print(DATABASE_PATH)

SQLite database created:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql\football_analytics.db


In [6]:
# ==========================================================
# CELL 4 — LOAD DATA INTO SQL TABLES
# ==========================================================

players.to_sql(
    "players",
    connection,
    if_exists="replace",
    index=False
)

teams.to_sql(
    "teams",
    connection,
    if_exists="replace",
    index=False
)

matches.to_sql(
    "matches",
    connection,
    if_exists="replace",
    index=False
)

print("Tables created successfully.")

Tables created successfully.


In [7]:
# ==========================================================
# CELL 5 — CHECK SQL TABLES
# ==========================================================

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)

,name
0,matches
1,players
2,teams


In [8]:
# ==========================================================
# CELL 6 — SQL QUERY: TOP SCORERS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    minutes
FROM players
WHERE goals > 0
ORDER BY goals DESC, assists DESC
LIMIT 20;
"""

top_scorers_sql = pd.read_sql_query(
    query,
    connection
)

display(top_scorers_sql)

,player,team,position,goals,assists,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,614.0
3,Erling Haaland,Norway,FW,7.0,0.0,465.0
4,Ousmane Dembélé,France,MF,6.0,2.0,592.0
5,Harry Kane,England,FW,6.0,1.0,652.0
6,Mikel Oyarzabal,Spain,FW,5.0,1.0,601.0
7,Vinicius Júnior,Brazil,FW,4.0,1.0,440.0
8,Julián Quiñones,Mexico,"FW,MF",4.0,1.0,410.0
9,Ismaila Sarr,Senegal,"MF,FW",4.0,1.0,364.0


In [9]:
# ==========================================================
# CELL 7 — SQL QUERY: TOP GOAL CONTRIBUTIONS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    goal_contributions,
    minutes
FROM players
ORDER BY goal_contributions DESC
LIMIT 20;
"""

top_contributors_sql = pd.read_sql_query(
    query,
    connection
)

display(top_contributors_sql)

,player,team,position,goals,assists,goal_contributions,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,14.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,12.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,8.0,614.0
3,Ousmane Dembélé,France,MF,6.0,2.0,8.0,592.0
4,Harry Kane,England,FW,6.0,1.0,7.0,652.0
5,Michael Olise,France,MF,0.0,7.0,7.0,646.0
6,Erling Haaland,Norway,FW,7.0,0.0,7.0,465.0
7,Bukayo Saka,England,MF,3.0,3.0,6.0,358.0
8,Mikel Oyarzabal,Spain,FW,5.0,1.0,6.0,601.0
9,Vinicius Júnior,Brazil,FW,4.0,1.0,5.0,440.0
